# Shannon-Prime VHT2 — Unified Test Suite

One-click test runner for Colab (T4/A100), RunPod (H100/B200), and local machines.

**What it tests:**
- Math invariants + falsification (Möbius, VHT2, squarefree, scaling law)
- Compression quality (bit allocation grid, head dim sweep, sequence stress)
- ComfyUI integration (nodes, lattice RoPE, Fisher weights, caching)
- Hardware matrix (CPU/CUDA parity, FP8, multi-GPU, dtype promotion)
- Cross-system (PrimePE×VHT2, edge cases, adversarial, thread safety)

Copyright (C) 2026 Ray Daniels. All Rights Reserved. Licensed under AGPLv3.

In [ ]:
# ── Step 1: Clone repos ──
import os, subprocess

BASE = '/content/shannon-prime-repos' if os.path.exists('/content') else os.path.expanduser('~/shannon-prime-repos')
os.makedirs(BASE, exist_ok=True)

REPOS = [
    ('shannon-prime', 'https://github.com/nihilistau/shannon-prime.git'),
    ('shannon-prime-engine', 'https://github.com/nihilistau/shannon-prime-engine.git'),
    ('shannon-prime-llama', 'https://github.com/nihilistau/shannon-prime-llama.git'),
    ('shannon-prime-comfyui', 'https://github.com/nihilistau/shannon-prime-comfyui.git'),
]

for name, url in REPOS:
    path = os.path.join(BASE, name)
    if os.path.exists(path):
        print(f'  [OK] {name} already cloned')
        subprocess.run(['git', '-C', path, 'pull', '--ff-only'], capture_output=True)
    else:
        print(f'  Cloning {name}...')
        subprocess.run(['git', 'clone', '--recursive', url, path])

print(f'\nAll repos at: {BASE}')

In [ ]:
# ── Step 2: Hardware Detection ──
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        tier = 'Blackwell' if p.major >= 10 else 'Ada/Hopper' if p.major >= 8 else 'Ampere' if p.major >= 8 else 'Older'
        print(f'  GPU {i}: {p.name} ({p.total_mem/1e9:.1f}GB, SM {p.major}.{p.minor}, {tier})')
        if hasattr(torch, 'float8_e4m3fn'):
            print(f'  FP8 support: yes')
else:
    print('  No CUDA — CPU-only tests will run')

In [ ]:
# ── Step 3: Run Full Test Suite ──
import sys, os, json

TESTS_DIR = os.path.join(BASE, 'tests') if os.path.exists(os.path.join(BASE, 'tests')) \
    else os.path.join(BASE, 'shannon-prime', 'tests')

# Try unified runner first
unified = os.path.join(BASE, 'tests', 'run_tests.py')
if os.path.exists(unified):
    os.chdir(os.path.dirname(unified))
    sys.path.insert(0, os.path.dirname(unified))
    !python run_tests.py --report both --verbose
else:
    print('Unified runner not found — running per-repo tests')
    sp_tests = os.path.join(BASE, 'shannon-prime', 'tests')
    for tf in ['test_torch.py', 'test_sqfree.py', 'test_comfyui.py']:
        path = os.path.join(sp_tests, tf)
        if os.path.exists(path):
            print(f'\n=== {tf} ===')
            !python {path}

In [ ]:
# ── Step 4: Display Report ──
import glob
from IPython.display import HTML, display, Markdown

results_dir = os.path.join(BASE, 'tests', 'results')
if os.path.exists(results_dir):
    # Find latest HTML report
    htmls = sorted(glob.glob(os.path.join(results_dir, 'report_*.html')))
    if htmls:
        with open(htmls[-1]) as f:
            display(HTML(f.read()))
    
    # Also show Markdown
    mds = sorted(glob.glob(os.path.join(results_dir, 'report_*.md')))
    if mds:
        with open(mds[-1]) as f:
            display(Markdown(f.read()))
    
    # Load JSON for programmatic access
    jsons = sorted(glob.glob(os.path.join(results_dir, 'run_*.json')))
    if jsons:
        with open(jsons[-1]) as f:
            results = json.load(f)
        print(f"\nJSON loaded: {results['summary']['passed']}/{results['summary']['total']} passed")
else:
    print('No results directory found')

In [ ]:
# ── Step 5: Quick Compression Benchmark (optional) ──
# Run just the compression suite with detailed output
if os.path.exists(unified):
    !python run_tests.py --suite compression --verbose